# Text extraction (T21) — MarkItDown -> EasyOCR fallback, on real documents

Exercises `classiflow.ingesta.extract.extract_document()` against two real municipal
PDFs from `playground/samples/`: one with a real text layer (MarkItDown handles it
directly) and one that's a genuine scan with no text layer at all (MarkItDown yields
nothing, so it falls through to the EasyOCR + pymupdf rendering path).

Also demonstrates the individual `MarkItDownExtractor` / `OCRExtractor` classes on
their own — each is independently testable/usable, not just through the chain.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.
>
> **Heads up**: the OCR section takes a few minutes on CPU (~3 min for a multi-page
> scan in local testing) — EasyOCR runs a real detection + recognition model per page,
> there's no shortcut around that on CPU hardware.


## 1 — Imports

In [1]:
from pathlib import Path

import classiflow
from classiflow.ingesta.extract import MIN_TEXT_FOR_OCR, MIN_USABLE_TEXT, extract_document
from classiflow.ingesta.extractors import MarkItDownExtractor, OCRExtractor

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"

print(f"MIN_TEXT_FOR_OCR = {MIN_TEXT_FOR_OCR}  (below this, MarkItDown's result triggers OCR)")
print(f'MIN_USABLE_TEXT  = {MIN_USABLE_TEXT}  (below this after OCR, extract_document returns "")')

MIN_TEXT_FOR_OCR = 50  (below this, MarkItDown's result triggers OCR)
MIN_USABLE_TEXT  = 20  (below this after OCR, extract_document returns "")


## 2 — A real text-based PDF: MarkItDown handles it, OCR never runs

`convenio_2_2013.pdf` has a real embedded text layer — MarkItDown extracts it
directly and the result is already well above `MIN_TEXT_FOR_OCR`, so the OCR stage
in the chain never executes.

In [2]:
text_pdf = (_SAMPLES_DIR / "convenio_2_2013.pdf").read_bytes()
print(f"file size: {len(text_pdf):,} bytes")

text = extract_document(text_pdf, "convenio_2_2013.pdf")
print(f"\nextracted: {len(text):,} chars")
print(f"preview  : {text[:300]!r}")

file size: 121,098 bytes

extracted: 6,814 chars
preview  : 'Decreto No 143712013\n\nConvenio N" ,..,,,Q?....~~~ _.- -...,...\n\n1 .k?. aal  Folio -49.. Tomo ..,i ,...\n\n!  Fecha .a 1 ..l..\n1\n!i I\n\nSecretaria de Gobierno - Direccton Gral. de Gobie.\n. .\n\nCONVENIO DE PESTAMO DE US  GRATUITO dwk%&died de bacio\n\n:,  2,.\n\nB\nEntre  la  MUNICIPALIDAD DE  ROSARIO,  repres'


## 3 — The individual extractor classes, directly

`extract_document()` is a thin orchestrator over a chain of `ExtractorBase`
implementations — each one is usable (and testable) completely on its own.

In [3]:
markitdown_extractor = MarkItDownExtractor()
direct_text = markitdown_extractor.extract(text_pdf, "convenio_2_2013.pdf")
matches = direct_text == text
print(f"MarkItDownExtractor directly: {len(direct_text):,} chars")
print(f"matches extract_document() result: {matches}")

MarkItDownExtractor directly: 6,814 chars
matches extract_document() result: True


## 4 — A genuine scan: MarkItDown finds nothing, OCR does the real work

`boletin_65_2005_doc_39153.pdf` has no text layer at all — this is where the chain
actually falls through to `OCRExtractor`, which renders each page via pymupdf at
`Settings.ocr_render_dpi` (200) DPI and runs EasyOCR on the resulting image.

This cell takes a few minutes on CPU — that's genuinely how long real OCR inference
takes per page, not a notebook artifact.

In [4]:
import time

scanned_pdf = (_SAMPLES_DIR / "boletin_65_2005_doc_39153.pdf").read_bytes()
print(f"file size: {len(scanned_pdf):,} bytes")

# Confirm MarkItDown alone really does come up empty on this one, before paying for OCR.
markitdown_only = markitdown_extractor.extract(scanned_pdf, "boletin_65_2005_doc_39153.pdf")
print(f"MarkItDown alone: {len(markitdown_only)} chars (below MIN_TEXT_FOR_OCR -> OCR will run)")

start = time.monotonic()
text = extract_document(scanned_pdf, "boletin_65_2005_doc_39153.pdf")
elapsed = time.monotonic() - start

print(f"\nOCR fallback took {elapsed:.1f}s")
print(f"extracted: {len(text):,} chars")
print(f"preview  : {text[:300]!r}")

file size: 1,636,073 bytes
MarkItDown alone: 0 chars (below MIN_TEXT_FOR_OCR -> OCR will run)

OCR fallback took 180.9s
extracted: 122,871 chars
preview  : 'ú?8 39\n175772402577 02 Rosnaij RESLSIEADo 28 Dic 2005 MA1SAG\'ALDrzL:NTRADAS AFCHIIVO GLENEHAL\nCONCEJO MUNICIAI ROSARIO Direccion {cnera} de Despacho\nLA MUNICIPALIDAD DE ROSARIO HA SANCIONADO LA SICUIENTE\n0 R D E N A NZ A (N" 7.948)\nCAPITULO  MODIFICACIONES AL CÓDIGO TRIBUTARIO MUNICIPAL\nArtículo 1".'


## 5 — The OCR extractor directly, and a corrupt-input failure

Same `OCRExtractor` used inside the chain above, called directly — and what happens
when it's handed something that isn't a valid PDF at all: `pymupdf.FileDataError` is
caught and re-raised as `classiflow`'s own `OcrError`, not left as a bare pymupdf
exception leaking out of this module.

In [5]:
from classiflow.ingesta.extractors.exceptions import OcrError

ocr_extractor = OCRExtractor()

try:
    ocr_extractor.extract(b"this is not a pdf", "not-a-pdf.pdf")
except OcrError as exc:
    print(f"caught OcrError, as expected: {exc}")

caught OcrError, as expected: OCR failed for 'not-a-pdf.pdf': Failed to open stream


## 6 — Guardrail: extract_document() never raises, always degrades to `""`

Feeding `extract_document()` itself the same garbage bytes — the chain catches the
`OcrError` internally and returns an empty string rather than propagating it, so a
single bad document can't crash the pipeline job that's ingesting it.

In [6]:
result = extract_document(b"this is not a pdf", "not-a-pdf.pdf")
print(f"result: {result!r}")
assert not result
print("confirmed: extract_document() degrades gracefully instead of raising")

2026-08-11 23:11:22.471 | WARNING  | classiflow.ingesta.extract:extract_document:24 - OCR failed for 'not-a-pdf.pdf': Failed to open stream


result: ''
confirmed: extract_document() degrades gracefully instead of raising
